# Data Generation on Apple Silicon

In [1]:
import importlib.metadata
import time
import json
from pathlib import Path

import torch
from tqdm import tqdm

In [2]:
# -----------------------------
# Environment checks
# -----------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    from mlx_lm import load, generate
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

Using device: mps


In [3]:
# -----------------------------
# Configuration
# -----------------------------
if torch.cuda.is_available():
    MLX_MODEL = "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF"
else:
    MLX_MODEL = "lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit"

print(f"Using model: {MLX_MODEL}")

NUM_PARAPHRASES = 1          # exactly one paraphrase per chunk
CHUNK_SIZE_TOKENS = 4096     # ~full context window
CHUNK_OVERLAP_TOKENS = 256   # small overlap for boundary safety
MAX_TOKENS = CHUNK_SIZE_TOKENS      # paraphrase length

INPUT_CORPUS = "Datasets/finance_bench_corpus.json"
OUTPUT_DIR = Path("paraphrase_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Using model: lmstudio-community/Qwen3-4B-Instruct-2507-MLX-8bit


In [ ]:
PARAPHRASE_PROMPT_TEMPLATE = """Use the information in the following snippet to write an informational paragraph in your own words.
Make sure to cover all the information, including all entities, dates and places in the original document.
Do not add additional material.
Directly output the paragraph and nothing else.

<document>
{doc}
</document>
"""

In [4]:
def mlx_generate_compat(model, tokenizer, prompt: str, max_tokens: int) -> str:
    """
    Call mlx_lm.generate with chat-template support.
    Uses DEFAULT decoding parameters.
    """
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
        )

    return generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=False,
    )

In [5]:
def chunk_text(tokenizer, text, chunk_size, overlap):
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start = end - overlap
        if start < 0:
            start = 0

    return chunks

In [6]:
def get_last_completed_chunk(path):
    """
    Returns the highest chunk_id already written to a JSONL file.
    If file doesn't exist, returns -1.
    """
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [7]:
with open(INPUT_CORPUS, "r") as f:
    corpus = json.load(f)

print(f"Loaded corpus with {len(corpus)} documents")


Loaded corpus with 24 documents


In [8]:
# Precompute all chunks for all documents (ensures consistency across all sections)
doc_chunks = {}
total_chunks = 0

for doc_idx, entry in enumerate(corpus):
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )
    doc_chunks[doc_idx] = chunks
    total_chunks += len(chunks)

print(f"Precomputed chunks for {len(corpus)} documents ({total_chunks} total chunks)")

NameError: name 'tokenizer' is not defined

In [9]:
print("Loading model...")
model, tokenizer = load(MLX_MODEL)
print("Model loaded.")

Loading model...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model loaded.


In [ ]:
doc_pbar = tqdm(corpus, desc="Documents")
for doc_idx, entry in enumerate(doc_pbar):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    full_doc = entry["text"]

    chunks = chunk_text(
        tokenizer,
        full_doc,
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS,
    )

    doc_pbar.set_postfix_str(f"{doc_name} ({len(chunks)} chunks)")

    output_path = OUTPUT_DIR / f"{doc_name}.jsonl"
    last_chunk = get_last_completed_chunk(output_path)
    start_chunk = last_chunk + 1
    
    if last_chunk >= 0:
        tqdm.write(f"Resuming {doc_name} from chunk {start_chunk}")

    with open(output_path, "a") as out_f:
        for chunk_idx in tqdm(range(start_chunk, len(chunks)), desc=f"Chunks", leave=False):
            chunk = chunks[chunk_idx]
            prompt = PARAPHRASE_PROMPT_TEMPLATE.format(doc=chunk)

            start_time = time.time()
            paraphrase = mlx_generate_compat(
                model,
                tokenizer,
                prompt,
                max_tokens=MAX_TOKENS,
            )
            elapsed = time.time() - start_time

            num_tokens = len(tokenizer.encode(paraphrase))

            record = {
                "doc_name": doc_name,
                "chunk_id": chunk_idx,
                "method": "paraphrase",
                "text": paraphrase,
                "tokens": num_tokens,
                "generation_time_sec": elapsed,
            }

            out_f.write(json.dumps(record) + "\n")
            out_f.flush()

    tqdm.write(f"Saved {doc_name} to {output_path}")

In [ ]:
SYNTHETIC_QA_PROMPT = """Generate a comprehensive list of fact-based questions and corresponding answers that can be answered explicitly from the document.

Requirements:
- Cover all entities, including people, organizations, dates, locations, quantities, and named concepts.
- Questions must be unambiguous, properly capitalized, and end with a question mark.
- Answers must be as concise as possible and use wording from the document when applicable.
- Output ONE question-answer pair per line.
- Separate the question and answer by a single space.
- Do NOT add any commentary or extra text.
- Do NOT invent information not present in the document.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_synthetic_qa(chunk, max_tokens=1024):
    prompt = SYNTHETIC_QA_PROMPT.format(chunk=chunk)
    text = mlx_generate_compat(model, tokenizer, prompt, max_tokens)
    return [l.strip() for l in text.split("\n") if l.strip()]

In [ ]:
def parse_qa_lines(lines):
    """
    Parse alternating question / answer lines.
    Returns list of (question, answer).
    """
    pairs = []
    i = 0
    while i + 1 < len(lines):
        q = lines[i].strip()
        a = lines[i + 1].strip()

        if q.endswith("?") and a:
            pairs.append((q, a))

        i += 2
    return pairs

In [ ]:
import os

QA_OUTPUT_DIR = Path("synthetic_qa_outputs")
QA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

doc_pbar = tqdm(corpus, desc="[QA] Documents")
for doc_idx, entry in enumerate(doc_pbar):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    full_doc = entry["text"]

    chunks = chunk_text(
        tokenizer,
        full_doc,
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS,
    )

    doc_pbar.set_postfix_str(f"{doc_name} ({len(chunks)} chunks)")

    output_path = QA_OUTPUT_DIR / f"{doc_name}.jsonl"
    last_chunk = get_last_completed_chunk(output_path)
    start_chunk = last_chunk + 1
    
    if last_chunk >= 0:
        tqdm.write(f"[QA] Resuming {doc_name} from chunk {start_chunk}")

    with open(output_path, "a") as out_f:
        for chunk_idx in tqdm(range(start_chunk, len(chunks)), desc=f"Chunks", leave=False):
            chunk = chunks[chunk_idx]
            qa_lines = generate_synthetic_qa(chunk)
            pairs = parse_qa_lines(qa_lines)

            for question, answer in pairs:
                record = {
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "question": question,
                    "answer": answer,
                    "method": "synthetic_qa",
                }

                out_f.write(json.dumps(record) + "\n")
                out_f.flush()
                os.fsync(out_f.fileno())

    tqdm.write(f"[QA] Saved {doc_name} to {output_path}")

In [10]:
D31_STRATEGY_PROMPT = """Consider the following document. 
What are some strategies specific to this document that I can use to help me learn and remember all of the information contained?

Requirements:
- Strategies should be specific to the structure and content of the document.
- Use markdown.
- Prefix each strategy with ##.
- Do NOT summarize the document.
- Do NOT repeat the document content verbatim.

<document>
{chunk}
</document>
"""

In [11]:
def generate_task_agnostic_strategies(chunk, max_tokens=512):
    prompt = D31_STRATEGY_PROMPT.format(chunk=chunk)
    text = mlx_generate_compat(model, tokenizer, prompt, max_tokens)
    return text.strip()

In [12]:
def split_strategies(strategy_text):
    blocks = strategy_text.split("##")
    return [
        "##" + b.strip()
        for b in blocks
        if b.strip()
    ]

In [13]:
ACTIVE_READING_PROMPT = """Here is a learning strategy:

{strategy}

Apply this strategy to the following document:

<document>
{chunk}
</document>
"""

In [14]:
def apply_active_reading(strategy, chunk, max_tokens=1024):
    prompt = ACTIVE_READING_PROMPT.format(
        strategy=strategy,
        chunk=chunk
    )
    return mlx_generate_compat(model, tokenizer, prompt, max_tokens)

In [15]:
def get_last_completed_chunk(path):
    if not path.exists():
        return -1

    last_chunk = -1
    with open(path, "r") as f:
        for line in f:
            try:
                obj = json.loads(line)
                last_chunk = max(last_chunk, obj.get("chunk_id", -1))
            except json.JSONDecodeError:
                continue
    return last_chunk

In [16]:
AR_DIR = Path("active_reading_outputs")
AR_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
from tqdm import tqdm
import os
import random
import json

for doc_idx, entry in enumerate(tqdm(corpus, desc="Active Reading documents")):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )

    out_path = AR_DIR / f"{doc_name}.jsonl"

    # checkpoint
    last_done = get_last_completed_chunk(out_path)
    if last_done >= 0:
        print(f"[AR resume] {doc_name}: skipping chunks 0–{last_done}")

    with open(out_path, "a") as f:
        for chunk_idx, chunk in enumerate(chunks):
            
            if chunk_idx % 50 == 0:
                print(f"{doc_name} | chunk {chunk_idx}")
            
            # skip completed chunks
            if chunk_idx <= last_done:
                continue

            # 1. generate strategies (D.3.1)
            strategy_text = generate_task_agnostic_strategies(chunk)

            # 2. split into individual strategies
            strategies = split_strategies(strategy_text)
            if not strategies:
                print(f"[AR warning] {doc_name} chunk {chunk_idx}: no strategies generated, skipping")
                continue

            # optional: limit strategies per chunk
            strategies = random.sample(
                strategies,
                k=min(3, len(strategies))
            )

            # 3. apply each strategy (D.3)
            for strat_idx, strategy in enumerate(strategies):
                ar_out = apply_active_reading(strategy, chunk)

                f.write(json.dumps({
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "strategy_id": strat_idx,
                    "strategy_type": "task_agnostic",
                    "strategy": strategy,
                    "active_reading": ar_out,
                    "method": "active_reading_d3"
                }) + "\n")

                f.flush()
                os.fsync(f.fileno())

Active Reading documents:   8%|▊         | 2/24 [00:00<00:04,  5.04it/s]

[AR resume] 3M_2018_10K: skipping chunks 0–36
3M_2018_10K | chunk 0
[AR resume] 3M_2023Q2_10Q: skipping chunks 0–21
3M_2023Q2_10Q | chunk 0


Active Reading documents:  17%|█▋        | 4/24 [00:00<00:04,  4.69it/s]

[AR resume] AES_2022_10K: skipping chunks 0–55
AES_2022_10K | chunk 0
AES_2022_10K | chunk 50
[AR resume] AMAZON_2019_10K: skipping chunks 0–17
AMAZON_2019_10K | chunk 0


Active Reading documents:  21%|██        | 5/24 [00:01<00:04,  4.61it/s]

[AR resume] AMCOR_2020_10K: skipping chunks 0–35
AMCOR_2020_10K | chunk 0
[AR resume] AMCOR_2023_10K: skipping chunks 0–21
AMCOR_2023_10K | chunk 0


Active Reading documents:  25%|██▌       | 6/24 [11:24<1:09:45, 232.50s/it]

AMD_2022_10K | chunk 0


Active Reading documents:  25%|██▌       | 6/24 [24:43<1:14:10, 247.24s/it]


KeyboardInterrupt: 

In [ ]:
D32_STRATEGY_PROMPT = """I need to study for a trivia competition.

Generate a list of questions that covers every piece of information in this document.
After generating all the questions, for each question, generate a general study strategy
or prompt that would help me memorize that kind of information (without focusing too much
on the particular question).

The prompt should outline a detailed set of guidelines or step-by-step instructions for
how I should rehearse or exercise the information to most effectively internalize it.

Output all the questions, then <start_strategies>, then all the strategies.
Prefix each strategy with ##.

<document>
{chunk}
</document>
"""

In [ ]:
def generate_task_specific_strategies(chunk, task_description, max_tokens=1024):
    prompt = D32_STRATEGY_PROMPT.format(
        chunk=chunk,
        task_description=task_description
    )
    text = mlx_generate_compat(model, tokenizer, prompt, max_tokens)
    return text.strip()

In [ ]:
def split_task_specific_strategies(text):
    if "<start_strategies>" not in text:
        return []

    _, strategy_block = text.split("<start_strategies>", 1)
    blocks = strategy_block.split("##")
    return [
        "##" + b.strip()
        for b in blocks
        if b.strip()
    ]

In [ ]:
from tqdm import tqdm
import os
import random
import json

for doc_idx, entry in enumerate(tqdm(corpus, desc="Active Reading D.3.2 documents")):
    doc_name = entry.get("doc_name", f"doc_{doc_idx}")
    chunks = chunk_text(
        tokenizer,
        entry["text"],
        CHUNK_SIZE_TOKENS,
        CHUNK_OVERLAP_TOKENS
    )

    out_path = AR_DIR / f"{doc_name}_task_specific.jsonl"
    last_done = get_last_completed_chunk(out_path)

    with open(out_path, "a") as f:
        for chunk_idx, chunk in enumerate(chunks):

            if chunk_idx <= last_done:
                continue

            # 1. generate task-specific strategies (D.3.2 prompt defines the task)
            ts_text = generate_task_specific_strategies(chunk)

            # 2. extract strategies only
            strategies = split_task_specific_strategies(ts_text)
            if not strategies:
                continue

            # optional: limit strategies per chunk
            strategies = random.sample(
                strategies,
                k=min(3, len(strategies))
            )

            # 3. apply each strategy (D.3)
            for strat_idx, strategy in enumerate(strategies):
                ar_out = apply_active_reading(strategy, chunk)

                f.write(json.dumps({
                    "doc_name": doc_name,
                    "chunk_id": chunk_idx,
                    "strategy_id": strat_idx,
                    "strategy_type": "task_specific",
                    "strategy": strategy,
                    "active_reading": ar_out,
                    "method": "active_reading_d3_2"
                }) + "\n")

                f.flush()
                os.fsync(f.fileno())